## Build `openalex_authors` by enriching `authors` with works data

#### Start from `authors`, enrich with work-level data from `openalex_works`

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.author_works_staging
AS
-- oxjob #1719: one profile on two seats of one work (a byline that prints the person twice, or two people joined on one
-- profile) counted the work twice in works_count, cited_by_count, h-index, counts_by_year and x_concepts, so /authors
-- read higher than /works?filter=author.id (A5130326380: 22 vs 16; 473,621 profiles on 17 Nov 2026). One row per seat
-- stays (raw names, affiliations and the name-evidence readers want every seat); is_first_seat flags each profile's
-- first seat on a work, and every per-work count in cell 10 reads only those.
-- repeat_seats lists the ~1.1M seats whose profile already sits on an earlier seat of the same work (only the ~518K of
-- 446M works that repeat a profile), read from a narrow scan of the author ids; the big seat scan below keeps its old
-- shape and only gains a broadcast lookup. (18 Nov: computing the check inside the big scan, with a lambda over the
-- whole authorships array, stopped Spark pruning the seat structs; shuffle went from ~560 GB to 1.3 TB and the
-- statement died of "GC overhead limit exceeded".)
-- Revert: drop repeat_seats, the join and is_first_seat here, and the is_first_seat terms in cell 10.
WITH repeat_seats AS (
  SELECT r.id AS work_id, p.pos AS seat_pos   -- 0-based position of a seat whose profile sits on an earlier seat
  FROM (
    SELECT w.id, w.authorships.author.id AS seat_ids
    FROM openalex.works.openalex_works w
    WHERE SIZE(ARRAY_DISTINCT(ARRAY_COMPACT(w.authorships.author.id))) < SIZE(ARRAY_COMPACT(w.authorships.author.id))
  ) r
  LATERAL VIEW EXPLODE(
    FILTER(SEQUENCE(0, SIZE(r.seat_ids) - 1), i -> ARRAY_POSITION(r.seat_ids, r.seat_ids[i]) != i + 1)
  ) p AS pos
),
seats AS (
  SELECT
    w.id,
    w.publication_year,
    w.publication_date,
    w.open_access,
    w.cited_by_count,
    w.counts_by_year,
    w.topics,
    w.concepts,
    FILTER(
      TRANSFORM(w.locations, loc -> loc.source),
      s -> s.id IS NOT NULL
    ) AS work_sources,
    w.created_date,
    w.updated_date,
    a.author,
    a.raw_author_name,
    a.institutions,
    a.author_order_number,
    a.is_corresponding,
    CAST(REPLACE(a.author.id, 'https://openalex.org/A', '') AS BIGINT) AS author_id_num,
    a_pos
  FROM openalex.works.openalex_works w
  LATERAL VIEW OUTER POSEXPLODE(w.authorships) AS a_pos, a
  WHERE a.author.id IS NOT NULL
    AND a.author.id != 'https://openalex.org/A9999999999'
    AND a.author.id != 'https://openalex.org/A5317838346'
    AND a.author.id != 'https://openalex.org/A5098778687'
    AND a.author.id != 'https://openalex.org/A5040317105'
),
exploded_works AS (
  SELECT /*+ BROADCAST(rs) */
    s.*,
    rs.work_id IS NULL AS is_first_seat
  FROM seats s
  LEFT JOIN repeat_seats rs ON rs.work_id = s.id AND rs.seat_pos = s.a_pos
)
SELECT
  ew.id                                                    AS work_id,
  COALESCE(ew.publication_year, YEAR(ew.publication_date)) AS pub_year,
  CAST(ew.open_access.is_oa AS BOOLEAN)                    AS is_oa,
  CAST(ew.cited_by_count AS INT)                           AS work_cited_by_count,
  ew.counts_by_year                                        AS work_counts_by_year,
  ew.topics                                                AS work_topics,
  ew.concepts                                              AS work_concepts,
  ew.work_sources                                          AS work_sources,
  ew.created_date                                          AS created_date,
  ew.updated_date                                          AS updated_date,
  CONCAT('https://openalex.org/A', CAST(ap.id AS STRING))  AS author_id,
  COALESCE(ew.author.display_name, ap.display_name)        AS author_display_name,
  ew.raw_author_name                                       AS raw_author_name,
  COALESCE(ew.institutions, ARRAY())                       AS author_institutions,
  ew.author_order_number                                   AS author_order_number,
  ew.is_corresponding                                      AS is_corresponding,
  ew.is_first_seat                                         AS is_first_seat   -- oxjob #1719; NULL on a no-works row
FROM openalex.authors.authors ap
LEFT JOIN exploded_works ew ON ap.id = ew.author_id_num;

-- Improve locality & skipping for the GROUP BY
OPTIMIZE openalex.authors.author_works_staging
ZORDER BY (author_id, pub_year);

-- Optional but helpful for planning & data skipping
ANALYZE TABLE openalex.authors.author_works_staging COMPUTE STATISTICS;

#### Create `author_topics`

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.author_topics CLUSTER BY (author_id) AS
WITH topics_and_share AS (
  WITH awt AS (
    -- explode once; distinct by (author, work, topic)
    SELECT DISTINCT
      author_id,
      work_id,
      CAST(t.id AS STRING) AS topic_id,
      t.display_name, t.subfield, t.field, t.domain, t.score
    FROM (
      SELECT author_id, work_id, EXPLODE_OUTER(work_topics) AS t
      FROM openalex.authors.author_works_staging
      WHERE work_id IS NOT NULL
    )
    WHERE t.id IS NOT NULL
  ),
  counts AS (
    -- author-topic counts AND topic totals in one pass
    SELECT
      author_id,                 -- NULL on total rows
      topic_id,
      MAX_BY(display_name, score) AS display_name,
      MAX_BY(subfield,     score) AS subfield,
      MAX_BY(field,        score) AS field,
      MAX_BY(domain,       score) AS domain,
      COUNT(DISTINCT work_id)    AS cnt
    FROM awt
    GROUP BY GROUPING SETS (
      (author_id, topic_id),
      (topic_id)
    )
  ),
  with_totals AS (
    -- compute topic_total via window BEFORE filtering author rows
    SELECT
      author_id,
      topic_id,
      display_name, subfield, field, domain,
      cnt AS topic_count,
      MAX(CASE WHEN author_id IS NULL THEN cnt END)
        OVER (PARTITION BY topic_id) AS topic_total
    FROM counts
  )
  -- final: include ALL topics; round share to 7 decimals; sort arrays
  SELECT
    author_id,
    ARRAY_SORT(
      COLLECT_LIST(
        STRUCT(
          topic_id AS id,
          display_name,
          CAST(topic_count AS INT) AS count,
          subfield,
          field,
          domain
        )
      ),
      (l, r) -> CASE WHEN l.count > r.count THEN -1 WHEN l.count < r.count THEN 1 WHEN l.id < r.id THEN -1 WHEN l.id > r.id THEN 1 ELSE 0 END
    ) AS topics,
    ARRAY_SORT(
      COLLECT_LIST(
        STRUCT(
          topic_id AS id,
          display_name,
          ROUND(CAST(topic_count AS DOUBLE) / NULLIF(topic_total, 0), 7) AS value,
          subfield,
          field,
          domain
        )
      ),
      (l, r) -> CASE WHEN l.value > r.value THEN -1 WHEN l.value < r.value THEN 1 WHEN l.id < r.id THEN -1 WHEN l.id > r.id THEN 1 ELSE 0 END
    ) AS topic_share
  FROM with_totals
  WHERE author_id IS NOT NULL
  GROUP BY author_id
)
SELECT * FROM topics_and_share;

OPTIMIZE openalex.authors.author_topics FULL;

In [ ]:
%sql
-- Snapshot current hashes before rebuild
CREATE OR REPLACE TABLE openalex.authors.openalex_authors_hash AS
SELECT id, updated_date,
  xxhash64(CONCAT_WS('|',
    CAST(id AS STRING),
    COALESCE(display_name, ''),
    COALESCE(full_name, ''),
    COALESCE(TO_JSON(raw_author_names), '[]'),
    COALESCE(orcid, ''),
    -- observed_orcids + ids are rebuilt here from parts so the pre-rebuild snapshot hashes like the
    -- post-rebuild row (cell 9): rows from before #1267 (NULL array, ids without the key) hash as
    -- observed_orcids = [orcid], which is what the rebuild produces when nothing else is observed.
    COALESCE(TO_JSON(COALESCE(observed_orcids, ARRAY_COMPACT(ARRAY(orcid)))), '[]'),
    COALESCE(TO_JSON(NAMED_STRUCT(
      'openalex', ids.openalex,
      'orcid', ids.orcid,
      'observed_orcids', COALESCE(observed_orcids, ARRAY_COMPACT(ARRAY(orcid)))
    )), '{}'),
    COALESCE(CAST(works_count AS STRING), '0'),
    COALESCE(CAST(cited_by_count AS STRING), '0'),
    COALESCE(TO_JSON(summary_stats), '{}'),
    COALESCE(TO_JSON(affiliations), '[]'),
    COALESCE(TO_JSON(last_known_institutions), '[]'),
    COALESCE(TO_JSON(sources), '[]'),
    COALESCE(TO_JSON(counts_by_year), '[]'),
    COALESCE(TO_JSON(topics), '[]'),
    -- topic_share excluded (non-deterministic: depends on global corpus)
    COALESCE(TO_JSON(x_concepts), '[]'),
    -- grants (oxjob #1847), as in cell 9. Needs the column on the live table before the first run (ALTER, one-off).
    CASE WHEN COALESCE(awards_count, 0) > 0 THEN CAST(awards_count AS STRING) END
  )) AS content_hash
FROM openalex.authors.openalex_authors;

### Create Final table `openalex.authors.openalex_authors`

In [ ]:
%sql
-- Guardrail: Abort if staging table is empty or drastically smaller than current
-- This prevents a cascading failure where empty openalex_authors causes NULL author.ids
-- in openalex_works, which then causes author_works_staging to be empty on the next run.

DECLARE OR REPLACE VARIABLE staging_count BIGINT;
DECLARE OR REPLACE VARIABLE current_count BIGINT;

SET VARIABLE staging_count = (SELECT COUNT(DISTINCT author_id) FROM openalex.authors.author_works_staging);
SET VARIABLE current_count = (SELECT COUNT(*) FROM openalex.authors.openalex_authors);

SELECT 
  staging_count AS staging_unique_authors,
  current_count AS current_authors,
  ROUND(staging_count * 100.0 / NULLIF(current_count, 0), 2) AS pct_of_current;

-- Fail if staging is empty
SELECT CASE 
  WHEN staging_count = 0 
  THEN RAISE_ERROR('GUARDRAIL FAILED: author_works_staging has 0 unique authors. This would create an empty openalex_authors table and cause a cascading failure.')
END;

-- Fail if staging has less than 90% of current (allows for some legitimate churn)
SELECT CASE 
  WHEN current_count > 0 AND staging_count < current_count * 0.9 
  THEN RAISE_ERROR(CONCAT('GUARDRAIL FAILED: staging has only ', CAST(ROUND(staging_count * 100.0 / current_count, 1) AS STRING), '% of current authors (', CAST(staging_count AS STRING), ' vs ', CAST(current_count AS STRING), '). This may indicate corrupted source data.'))
END;

In [0]:
%sql
-- oxjob #2106: the one normaliser for served entity names (author profiles, sources). Elsevier's dotless i + accent
-- ('Valentı́n Fuster', 8.5K profiles), decomposed accents (18K), Turkish i + dot above, invisible format characters
-- (soft hyphen, ZWSP, LRM/RLM, BOM, word joiner: 11.6K profiles; 'IODP Bre\xadmen Core Re\xadpos\xadit\xadory'), and for
-- sources HTML entities ('&amp;', '&#35745;...'). Unicode NFC is the canonical form of the same text. ZWNJ/ZWJ stay
-- (Persian, Indic). Mojibake is left alone (its soft hyphen is a byte of a broken letter). Name search (authors index:
-- asciifolding) and autocomplete missed these names: display_name.search found 0/100 (9 Nov 2026).
-- Source of truth and tests: oxjobs #2106 work/name_text.py, work/test_name_text.py (this body is generated from it by
-- work/patch_walden.py). The same body is created in authors/CreateAuthors, sources/CreateSourcesApi and
-- end2end/CreateWorksBase. Revert: git revert the #2106 commit (the next runs serve the stored names again).
CREATE OR REPLACE FUNCTION openalex.common.normalize_name_text(s STRING, decode_entities BOOLEAN)
RETURNS STRING
LANGUAGE PYTHON
DETERMINISTIC
COMMENT 'oxjob #2106: served entity name text normalised: dotless i/j + mark, Turkish i + dot above, invisible format characters removed (ZWNJ/ZWJ kept), NFC, HTML entities decoded when decode_entities. Returns its input when there is nothing to fix. Source + tests: oxjobs #2106 work/name_text.py'
AS $$
import html
import re
import unicodedata
from html.entities import html5 as _HTML5

_DOTLESS = re.compile('([\u0131\u0237])(?=[\u0300-\u036f])')
_TURKISH_I = re.compile('(?<=[i\u0130])\u0307+')  # 'i\u0307' -> 'i', '\u0130\u0307' -> '\u0130': the letter already has its dot
_ZWSP = re.compile('\u200b+')
_INVISIBLE = re.compile('[\u00ad\u200b\u200e\u200f\ufeff\u2060]')
_ENTITY = re.compile(r'&(#[0-9]{2,7}|#[xX][0-9a-fA-F]{2,6}|[A-Za-z][A-Za-z0-9]{1,30})\x3b')  # \x3b = semicolon: a literal one would split the SQL cell
_MOJIBAKE = re.compile('[\u00c3\u00c2\u00c4\u00c5\u00d0\u00d1][\u0080-\u00bf]|\u00e2\u20ac')
_SPACES = re.compile(r'\s+')
_CONTROL = re.compile('[\x00-\x1f\x7f-\x9f]')


_CODECS = ('latin-1', 'cp1252', 'cp1250', 'cp1251')


def _zwsp(m):
    s, i, j = m.string, m.start(), m.end()
    return ' ' if 0 < i and j < len(s) and s[i - 1].islower() and s[j].isupper() else ''


def _soft_hyphen_is_a_byte(s):
    """True when some U+00AD is byte 0xAD of a UTF-8 letter read as a one-byte codepage: 'GarcÃ\\xada' (latin-1),
    'Р\\xad.' (cp1251 'Э'), 'MedellĂ\\xadN' (cp1250), Thai 'à¸\\xad'. 'Cí\\xadcero' is not: 0xED needs two more bytes."""
    for i, ch in enumerate(s):
        if ch != '\u00ad':
            continue
        for codec in _CODECS:
            for k in (1, 2, 3):  # lead byte k chars back, then k continuation bytes ending at or after the 0xAD
                if i - k < 0:
                    continue
                try:
                    b = s[i - k:i + 1].encode(codec) + s[i + 1:i + 1 + (3 - k)].encode(codec)
                except UnicodeEncodeError:
                    continue
                if len(b) <= k:
                    continue
                n = 2 if 0xC2 <= b[0] <= 0xDF else 3 if 0xE0 <= b[0] <= 0xEF else 4 if 0xF0 <= b[0] <= 0xF4 else 0
                if n == k + 1 or (n > k + 1 and len(b) >= n):
                    try:
                        b[:n].decode('utf-8')
                        return True
                    except UnicodeDecodeError:
                        pass
    return False


def _decode_entity(m):
    """One well-formed '&name' + semicolon or '&#...' + semicolon token. html.unescape on a whole string also decodes
    references with no semicolon ('Planning &regional' -> 'Planning ®ional', #2107), so: numeric tokens through
    html.unescape (its cp1252 and invalid-codepoint rules), named ones only when the whole name is an HTML5 entity."""
    g = m.group(1)
    if g[0] == '#':
        return html.unescape(m.group(0))
    return _HTML5.get(g + '\x3b', m.group(0))


def normalize_name_text(s, decode_entities=False):
    if s is None:
        return None
    if _MOJIBAKE.search(s) or ('\u00ad' in s and _soft_hyphen_is_a_byte(s)):
        return s
    t = s
    if decode_entities and _ENTITY.search(t):
        for _ in range(3):  # double-encoded entities, bounded
            u = _ENTITY.sub(_decode_entity, t)
            if u == t:
                break
            t = u
    t = _DOTLESS.sub(lambda m: 'i' if m.group(1) == '\u0131' else 'j', t)
    t = _TURKISH_I.sub('', t)
    t = _ZWSP.sub(_zwsp, t)
    t = _INVISIBLE.sub('', t)
    t = unicodedata.normalize('NFC', t)
    if t == s:
        return s
    if not _CONTROL.search(s):  # a catalogue string's line breaks are #1912's to read: leave its whitespace
        t = _SPACES.sub(' ', t).strip()
    return t if t.strip() else s


return normalize_name_text(s, decode_entities)
$$


In [0]:
%sql
-- oxjob #2106: the profile names to serve normalised, built every run from every name the main statement below can
-- serve (a curated name, a served full_name, the mint-time display_name). The RLIKE gate is a superset of what the
-- function changes, so the Python function runs on ~100K names, not the 96M profiles. Only changed names kept.
-- Revert: remove this cell, the function cell above it, and the #2106 COALESCE + join in the final SELECT below.
CREATE OR REPLACE TABLE openalex.authors.author_name_text_fixes AS
WITH names AS (
  SELECT DISTINCT n FROM (
    SELECT NULLIF(curated_display_name, '') AS n FROM openalex.authors.author_names_curations
    UNION ALL
    SELECT TRIM(full_name) AS n FROM openalex.authors.authors
    UNION ALL
    SELECT display_name AS n FROM openalex.authors.authors
  )
  WHERE n RLIKE '[\\x{0300}-\\x{036F}\\x{00AD}\\x{200B}\\x{200E}\\x{200F}\\x{FEFF}\\x{2060}]'
)
SELECT raw_name, clean_name FROM (
  SELECT n AS raw_name, openalex.common.normalize_name_text(n, FALSE) AS clean_name FROM names
) WHERE clean_name IS DISTINCT FROM raw_name

In [0]:
%sql
-- oxjob #2346: a profile named with a leading title or a library-catalogue string serves the name its bylines show
-- ('Dr Jay Ebert' -> 'Jay Ebert'; 'Berni, Francesco, 1497-1535' -> 'Francesco Berni'). CreateWorkAuthorships serves
-- every byline name through openalex.authors.clean_byline_name; this serves the same function's output on the profile.
-- The gate is the fixes cell's: its title part (titles, 214,265 profiles, 17 Nov 2026) and, since 18 Nov (#2346 Part 2,
-- ~165K profiles), its catalogue part, minus multi-heading strings joined by ';' (the cleaner can garble those).
-- A catalogue rename changes the name the nightly rematch reads, so profiles whose rename would newly flag a seat that
-- is not the owner's are held (author_name_clean_holds, final SELECT), and the owner seats it would flag are pinned
-- (author_rematch_pins, reason oxjob2346-catalogue-name-owner-20261118). Built every run from every name the main statement
-- below can serve (the mint-time display_name, a served full_name, and #2106's normalised text of either). Only changed
-- names. The final SELECT cleans the #2106-normalised name, so a name gets both fixes, and never cleans a curated name.
-- shortcut: a #1990 marker_fixed_name (a seat's own spelling) is not in the list; it rarely starts with a title.
-- Revert: remove this cell and the #2346 COALESCE term + two joins in the final SELECT below.
CREATE OR REPLACE TABLE openalex.authors.author_byline_name_fixes AS
WITH names AS (
  SELECT DISTINCT n FROM (
    SELECT TRIM(full_name) AS n FROM openalex.authors.authors
    UNION ALL
    SELECT display_name AS n FROM openalex.authors.authors
    UNION ALL
    SELECT clean_name AS n FROM openalex.authors.author_name_text_fixes   -- #2106's text fix first, then this
  )
  WHERE (n RLIKE '(?-i:(^|,\\s*)\\(?(Dr|Dra|Drs|Prof|Professor|Mr|Ms|Miss|dr|prof)([-.\\s\\x{00A0})]|$))'
         AND NOT n RLIKE '[\\x{0000}-\\x{001F}\\x{007F}-\\x{009F}]|(?i)(https?://|www\\.|id\\.loc\\.gov|viaf\\.org/|orcid\\.org/|lattes\\.cnpq\\.br/|id_orcid)|(^|\\s)880-[0-9]{2}|[0-9]{3,4}\\??[\\s\\x{00A0}]*-[\\s\\x{00A0}]*([0-9]{3,4}\\??)?[\\s\\x{00A0}]*[.,)]?[\\s\\x{00A0}]*$|[0-9]{3,4}\\??\\s*-\\s*([0-9]{3,4}\\??)?\\s+\\p{L}|\\S[\\s\\x{00A0}]{3,}\\S|[0-9]{4}- ?[0-9]{4}-[0-9]{4}-[0-9]{3}[0-9X]|\\.\\s*(auteur|éditeur|traducteur|illustrateur|préfacier|compositeur|librettiste|graveur|photographe|cartographe|imprimeur|libraire|dessinateur|annotateur|commentateur|rédacteur)')
     OR (n RLIKE '[\\x{0000}-\\x{001F}\\x{007F}-\\x{009F}]|(?i)(https?://|www\\.|id\\.loc\\.gov|viaf\\.org/|orcid\\.org/|lattes\\.cnpq\\.br/|id_orcid)|(^|\\s)880-[0-9]{2}|[0-9]{3,4}\\??[\\s\\x{00A0}]*-[\\s\\x{00A0}]*([0-9]{3,4}\\??)?[\\s\\x{00A0}]*[.,)]?[\\s\\x{00A0}]*$|[0-9]{3,4}\\??\\s*-\\s*([0-9]{3,4}\\??)?\\s+\\p{L}|\\S[\\s\\x{00A0}]{3,}\\S|[0-9]{4}- ?[0-9]{4}-[0-9]{4}-[0-9]{3}[0-9X]|\\.\\s*(auteur|éditeur|traducteur|illustrateur|préfacier|compositeur|librettiste|graveur|photographe|cartographe|imprimeur|libraire|dessinateur|annotateur|commentateur|rédacteur)'
         AND NOT (n RLIKE ';' AND NOT n RLIKE '(?i)id_orcid|orcid\\.org/|;\\s*#\\s*[0-9]'))
)
SELECT raw_name, clean_name FROM (
  SELECT n AS raw_name, openalex.authors.clean_byline_name(n) AS clean_name FROM names
) WHERE clean_name IS DISTINCT FROM raw_name

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.openalex_authors
CLUSTER BY (id, updated_date) AS
WITH 
-- concepts (group by concept.id only, use MAX for score to ensure determinism)
author_concepts_exploded AS (
  SELECT
    author_id,
    concept.id AS concept_id,
    ANY_VALUE(concept.wikidata) AS wikidata,
    ANY_VALUE(concept.display_name) AS display_name,
    ANY_VALUE(concept.level) AS level,
    MAX(concept.score) AS score,
    COUNT(*) AS concept_count
  FROM openalex.authors.author_works_staging
  LATERAL VIEW OUTER EXPLODE(work_concepts) t AS concept
  WHERE concept.id IS NOT NULL
    AND is_first_seat   -- oxjob #1719: count works, not seats
  GROUP BY author_id, concept.id
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY author_id
    ORDER BY concept_count DESC, CAST(concept_id AS STRING) ASC
  ) <= 5
),
concepts_packed AS (
  SELECT
    author_id,
    ARRAY_SORT(
      COLLECT_LIST(
        STRUCT(
          concept_id AS id,
          wikidata,
          display_name,
          CAST(level AS INT),
          score,
          CAST(concept_count AS INT) AS count
        )
      ),
      (l, r) -> CASE WHEN l.score > r.score THEN -1 WHEN l.score < r.score THEN 1 WHEN l.id < r.id THEN -1 WHEN l.id > r.id THEN 1 ELSE 0 END
    ) AS x_concepts
  FROM author_concepts_exploded
  GROUP BY author_id
),

-- affiliations
affiliations_exploded AS (
  SELECT
    author_id,
    i.id                             AS inst_id,
    ANY_VALUE(i.ror)                 AS ror,
    ANY_VALUE(i.display_name)        AS display_name,
    ANY_VALUE(i.country_code)        AS country_code,
    ANY_VALUE(i.type)                AS type,
    ANY_VALUE(i.lineage)             AS lineage,
    SORT_ARRAY(COLLECT_SET(pub_year), false) AS years
  FROM (
    SELECT author_id, pub_year, EXPLODE_OUTER(author_institutions) AS i
    FROM openalex.authors.author_works_staging
    WHERE pub_year IS NOT NULL
  )
  WHERE i.id IS NOT NULL
  GROUP BY author_id, i.id
),
affiliations_packed AS (
  SELECT
    author_id,
    ARRAY_SORT(
      COLLECT_LIST(
        STRUCT(
          STRUCT(inst_id as id, ror, display_name, country_code, type, lineage) as institution,
          years
        )
      ),
      (l, r) -> CASE WHEN l.institution.id < r.institution.id THEN -1 WHEN l.institution.id > r.institution.id THEN 1 ELSE 0 END
    ) AS affiliations
  FROM affiliations_exploded
  GROUP BY author_id
),
sources_agg AS (
  SELECT
    author_id,
    -- COLLECT_SET deduplicates the struct objects
    ARRAY_SORT(COLLECT_SET(src), (l, r) -> CASE WHEN l.id < r.id THEN -1 WHEN l.id > r.id THEN 1 ELSE 0 END) AS sources
  FROM (
    SELECT author_id, EXPLODE(work_sources) AS src
    FROM openalex.authors.author_works_staging
    WHERE work_sources IS NOT NULL
  ) t
  WHERE src.id IS NOT NULL
  GROUP BY author_id
),

-- counts_by_year (oxjob #1878): works_count / oa_works_count by publication year; cited_by_count = the citations
-- these works RECEIVED in that year, summed from each work's own counts_by_year (citing years start in 2012),
-- the pre-Walden meaning and the same as works. No year after the current one (10000 = no year, Buddhist-era 25xx).
works_per_year AS (
  SELECT
    author_id,
    CAST(pub_year AS INT) AS year,
    CAST(count(*) AS INT) AS works_count,
    CAST(sum(if(is_oa, 1, 0)) AS INT) AS oa_works_count
  FROM openalex.authors.author_works_staging
  WHERE pub_year IS NOT NULL AND pub_year <= YEAR(current_date())
    AND is_first_seat   -- oxjob #1719: count works, not seats
  GROUP BY author_id, pub_year
),
citations_per_year AS (
  SELECT p.author_id, cy.year AS year, CAST(SUM(cy.cited_by_count) AS INT) AS cited_by_count
  FROM (
    -- one row per (author, work): a repeated seat must not count a work's citations twice (oxjob #1719's flag)
    SELECT author_id, work_counts_by_year
    FROM openalex.authors.author_works_staging
    WHERE is_first_seat
  ) p
  LATERAL VIEW EXPLODE(p.work_counts_by_year) e AS cy
  WHERE cy.year <= YEAR(current_date()) AND cy.cited_by_count > 0
  GROUP BY p.author_id, cy.year
),
counts_per_year AS (
  SELECT
    COALESCE(w.author_id, c.author_id) AS author_id,
    COALESCE(w.year, c.year) AS year,
    COALESCE(w.works_count, 0) AS works_count,
    COALESCE(w.oa_works_count, 0) AS oa_works_count,
    COALESCE(c.cited_by_count, 0) AS cited_by_count
  FROM works_per_year w
  FULL OUTER JOIN citations_per_year c ON w.author_id = c.author_id AND w.year = c.year
),
counts_agg AS (
  SELECT
    author_id,
    SORT_ARRAY(
      COLLECT_LIST(
        STRUCT(
          year,
          works_count,
          oa_works_count,
          cited_by_count
        )
      ),
      true
    ) as counts_by_year
  FROM counts_per_year
  GROUP BY author_id
),

-- last known institutions
last_inst_agg AS (
  SELECT
    wa.author_id,
    ARRAY_SORT(COLLECT_SET(STRUCT(i.id, i.ror, i.display_name, i.country_code, i.type, i.lineage)), (l, r) -> CASE WHEN l.id < r.id THEN -1 WHEN l.id > r.id THEN 1 ELSE 0 END) as last_known_institutions
  FROM openalex.authors.author_works_staging wa
  INNER JOIN (
      -- oxjob #1611: the latest year whose works name an institution for this author, not the latest year of all.
      -- A newest year whose seats carry no institution (a parser that left the seat bare, a review, a catalogue
      -- record) gave an empty list though older works name one: 6,443,688 profiles on 15 Nov 2026 (ZD 4171, 26540).
      SELECT author_id, MAX(pub_year) as max_year
      FROM openalex.authors.author_works_staging
      WHERE SIZE(FILTER(author_institutions, i -> i.id IS NOT NULL)) > 0
      GROUP BY author_id
  ) my ON wa.author_id = my.author_id AND wa.pub_year = my.max_year
  LATERAL VIEW OUTER EXPLODE(author_institutions) t AS i
  WHERE i.id IS NOT NULL
  GROUP BY wa.author_id
),

-- grants (oxjob #1847): awards that list this profile on any investigator slot (lead, co-lead or investigators[]) of the
-- published awards_api: the same people as investigators_full, read from their own author fields. Read through to_json, so this
-- build never depends on the field existing: before Nightly Grants' part 2 has published (or after a failed night) every count is 0.
awards_agg AS (
  SELECT author_id, CAST(COUNT(DISTINCT award_id) AS INT) AS awards_count
  FROM (SELECT a.id AS award_id, get_json_object(to_json(p), '$.author.id') AS author_id
        FROM openalex.awards.awards_api a
        LATERAL VIEW EXPLODE(concat(array(a.lead_investigator, a.co_lead_investigator), coalesce(a.investigators, array()))) t AS p)
  WHERE author_id IS NOT NULL
  GROUP BY author_id
),

-- main aggregation
main_agg_pre AS (
  SELECT
    CAST(REPLACE(b.author_id, 'https://openalex.org/A', '') AS BIGINT) AS id,
    b.author_id AS author_id_str,
    SLICE(
      ARRAY_SORT(
        ARRAY_DISTINCT(
          ARRAY_COMPACT(
            CONCAT(COLLECT_LIST(b.author_display_name), COLLECT_LIST(b.raw_author_name))
          )
        )
      ),
      1, 100
    ) AS raw_author_names,
    -- oxjob #1719: every per-work value reads the profile's first seat on each work (is_first_seat), so a work the
    -- profile holds two seats on counts once; raw_author_names above still reads every seat.
    CAST(COUNT_IF(b.is_first_seat) AS INT) AS works_count,
    COALESCE(CAST(SUM(IF(b.is_first_seat, work_cited_by_count, NULL)) AS INT), 0) AS cited_by_count,
    COUNT_IF(is_oa AND b.is_first_seat) AS oa_works_count,
    -- 2yr_mean_citedness (oxjob #1851, Kyle 26 Oct): the journal-impact-factor-style value the docs describe. With Y the
    -- current year: the citations received in Y-1 (each work's counts_by_year) by the works published in Y-3 and Y-2,
    -- divided by the number of those works (in 2026: 2025 citations to 2023-2024 works). Was the mean all-time
    -- cited_by_count of the works from Y-2 to Y.
    CAST(AVG(CASE WHEN b.is_first_seat AND b.pub_year BETWEEN YEAR(current_date()) - 3 AND YEAR(current_date()) - 2
                  THEN COALESCE(AGGREGATE(FILTER(b.work_counts_by_year, x -> x.year = YEAR(current_date()) - 1),
                                          CAST(0 AS BIGINT), (acc, x) -> acc + COALESCE(CAST(x.cited_by_count AS BIGINT), 0)), 0) END) AS DOUBLE) AS two_year_mean,
    CAST(COUNT_IF(b.is_first_seat AND work_cited_by_count >= 10) AS INT) AS i10_index,
    SORT_ARRAY(
      TRANSFORM(
        FILTER(COLLECT_LIST(IF(b.is_first_seat, work_cited_by_count, NULL)), x -> x IS NOT NULL),
        x -> CAST(x AS INT)
      ),
      false
    ) AS sorted_citations,
    MAX(b.pub_year) AS max_pub_year,
    CONCAT('https://api.openalex.org/works?filter=author.id:',
           REGEXP_EXTRACT(b.author_id, '/(A[0-9]+)$', 1)) AS works_api_url,
    CAST(NULL AS TIMESTAMP) AS updated_date,
    DATE_TRUNC('SECOND', CAST(MIN(b.created_date) AS TIMESTAMP)) AS created_date
  FROM openalex.authors.author_works_staging b
  GROUP BY b.author_id
),

-- trusted seat ORCIDs per profile (CreateAuthorObservedOrcids), minus a curated removal (#1267)
observed_pre AS (
  SELECT m.id,
    FILTER(COALESCE(oo.seat_orcids, ARRAY()), x -> NOT (x.orcid <=> NULLIF(oc.removed_orcid, ''))) AS seat_orcids_kept
  FROM main_agg_pre m
  LEFT JOIN openalex.authors.author_observed_orcids oo ON m.id = oo.author_id
  LEFT JOIN openalex.authors.author_orcid_curations oc ON m.id = oc.author_id
),

-- served name (oxjob #1297): serve the matching name (authors.full_name) instead of the mint-time display_name
-- when it is attested on the profile's own works under the nightly UpdateAuthorFullNames evidence rules
-- (>= 2 seats and is_clean: natural order, no digits/commas/credentials/doubled tokens) and either
--   (a) the two names disagree on the person's block (different surname or first initial) and full_name is at
--       least as well attested as display_name -- the PubMed 'Lastname Xy' junk displays that pollution_reset fixed; or
--   (b) full_name is a strict refinement of display_name: first initial -> word, middle appears or initial -> word,
--       nothing degrades (mirrors the enrich gain rule).
-- Never a full_name holding a BnF catalogue role phrase (oxjob #1952): the mint's first-comma flip 'Hilarion. Auteur du
-- texte Meynard' parses as a middle name 'Auteur du texte' and passed (b) on 182 profiles (2 Nov 2026). (?iu): Java's
-- (?i) folds ASCII only, so 'Éditeur' needs the u flag (10 profiles).
-- Otherwise the mint-time display_name stands. Catalog-order and other unclean legacy full_names (~8% of profiles)
-- are never served; a curation always wins (enriched_agg). Measured 2026-09-21: 1.67M of 128M active profiles change.
served_name AS (
  SELECT id,
    CASE WHEN f_n >= 2 AND f_clean AND NOT f_role AND f_last IS NOT NULL AND COALESCE(f_first, '') <> ''
          AND (
            (diff_block AND f_n >= COALESCE(d_n, 0))
            OR (NOT diff_block
                AND NOT (dfl = ffl AND dml = fml)
                AND (dfl = ffl OR (LENGTH(dfl) <= 1 AND LENGTH(ffl) >= 2))
                AND (dml = fml OR dml = '' OR (LENGTH(dml) = 1 AND SUBSTRING(fml, 1, 1) = dml)))
          )
         THEN full_name END AS served_name
  FROM (
    SELECT ar.id, TRIM(ar.full_name) AS full_name,
      fn.parsed_name.first AS f_first, fn.parsed_name.last AS f_last,
      fe.n AS f_n, fe.is_clean AS f_clean, de.n AS d_n,
      TRIM(ar.full_name) RLIKE '(?iu)\\.\\s*(auteur|autrice|éditeur|traducteur|illustrateur|préfacier|compositeur|librettiste|graveur|photographe|cartographe|imprimeur|libraire|dessinateur|annotateur|commentateur|rédacteur|destinataire|réalisateur|scénariste|metteur en scène)\\b' AS f_role,
      (dn.parsed_name.last IS NULL OR dn.parsed_name.last <> fn.parsed_name.last
       OR SUBSTRING(dn.parsed_name.first, 1, 1) <> SUBSTRING(fn.parsed_name.first, 1, 1)) AS diff_block,
      REGEXP_REPLACE(COALESCE(dn.parsed_name.first, ''),  '[^\\p{L}]', '') AS dfl,
      REGEXP_REPLACE(COALESCE(fn.parsed_name.first, ''),  '[^\\p{L}]', '') AS ffl,
      REGEXP_REPLACE(COALESCE(dn.parsed_name.middle, ''), '[^\\p{L}]', '') AS dml,
      REGEXP_REPLACE(COALESCE(fn.parsed_name.middle, ''), '[^\\p{L}]', '') AS fml
    FROM openalex.authors.authors ar
    LEFT JOIN openalex.authors.author_names fn ON TRIM(ar.full_name) = fn.raw_author_name
    LEFT JOIN openalex.authors.author_names dn ON TRIM(ar.display_name) = dn.raw_author_name
    LEFT JOIN openalex.authors.author_full_name_evidence fe
      ON fe.author_id = ar.id AND fe.raw = TRIM(ar.full_name) AND fe.n IS NOT NULL
    LEFT JOIN openalex.authors.author_full_name_evidence de
      ON de.author_id = ar.id AND de.raw = TRIM(ar.display_name) AND de.n IS NOT NULL
    WHERE ar.full_name IS NOT NULL AND TRIM(ar.full_name) <> ''
      AND TRIM(ar.full_name) IS DISTINCT FROM TRIM(ar.display_name)
  )
),

-- oxjob #1990: a served name that kept a paper's affiliation superscript as its last token ('Albert Hofman D',
-- 'Giorgio Parisi A', 'Maurizio Fava M. D'; ZD 26847) serves the profile's own most-attested spelling without it, when
-- that spelling is on >= 2 of the profile's current seats and on >= 10x as many as the served string. The letter came
-- from the byline the profile was minted from, mostly via full_name, which #1297's served-name rule then serves
-- (Hofman: 0 of his seats carry 'Albert Hofman D'). A real name form keeps its letter because its own seats carry it
-- ('Rafael Ruiz S' 18/18, 'Julio César Alonso C' 1012/1012). Candidates are both names a profile can serve
-- (display_name, full_name); enriched_agg joins the one it would serve. Measured 2026-11-16 on Databricks:
-- 2,367 profiles change (2,047 with > 50 works; 0 of the 5 checked real name forms), ~562K works.
-- Revert: delete this CTE, its LEFT JOIN and the mn.marker_fixed_name term in enriched_agg.
marker_name AS (
  SELECT id, nm, MAX_BY(raw, STRUCT(is_cased, n, raw)) AS marker_fixed_name
  FROM (
    SELECT ar.id, ar.nm, e.raw, e.n,
      (e.raw RLIKE '\\p{Ll}' AND e.raw NOT RLIKE '\\p{Lu}{4,}') AS is_cased,   -- 'Albert Hofman' before 'ALBERT HOFMAN'
      TRIM(LOWER(REGEXP_REPLACE(e.raw, '[^\\p{L}]+', ' '))) = ar.k_base AS is_base,
      SUM(CASE WHEN TRIM(LOWER(REGEXP_REPLACE(e.raw, '[^\\p{L}]+', ' '))) = ar.k_base THEN e.n ELSE 0 END)
        OVER (PARTITION BY ar.id, ar.nm) AS base_n,
      SUM(CASE WHEN TRIM(LOWER(REGEXP_REPLACE(e.raw, '[^\\p{L}]+', ' '))) = ar.k_dn THEN e.n ELSE 0 END)
        OVER (PARTITION BY ar.id, ar.nm) AS dn_n
    FROM (
      SELECT DISTINCT id, nm,
        TRIM(LOWER(REGEXP_REPLACE(nm, '[^\\p{L}]+', ' '))) AS k_dn,
        TRIM(LOWER(REGEXP_REPLACE(REGEXP_REPLACE(nm, ' ([A-Z]|M\\. D)$', ''), '[^\\p{L}]+', ' '))) AS k_base
      FROM openalex.authors.authors
      LATERAL VIEW EXPLODE(ARRAY(display_name, TRIM(full_name))) x AS nm
      WHERE nm RLIKE '^[^ ,]{2,}( [^ ,]{2,})+ ([A-Z]|M\\. D)$'
    ) ar
    JOIN openalex.authors.author_full_name_evidence e ON e.author_id = ar.id AND e.n IS NOT NULL
  )
  WHERE is_base AND base_n >= 2 AND base_n >= 10 * dn_n
    AND raw NOT RLIKE '[0-9,;:()\\[\\]*]'
  GROUP BY id, nm
),

-- join authors for identity fields; apply display_name curation override, then the marker rule (#1990), then the served-name rule (#1297)
enriched_agg AS (
  SELECT
    m.*,
    COALESCE(NULLIF(c.curated_display_name, ''), mn.marker_fixed_name, sn.served_name, ar.display_name) AS display_name,
    -- primary ORCID (#1267): curated > most trusted seats > legacy mint-time value (authors.orcid, never recomputed).
    -- The legacy value is kept only while the profile has works: an emptied duplicate must not hold a person's
    -- ORCID forever (77,613 zero-works profiles did on 2026-09-23; these profiles are slated for deletion).
    CASE
      WHEN oc.curated_orcid IS NOT NULL AND oc.curated_orcid <> '' THEN oc.curated_orcid  -- replace: set correct ORCID (#410)
      WHEN SIZE(op.seat_orcids_kept) > 0 THEN op.seat_orcids_kept[0].orcid                -- organic: the trusted seat ORCID with the most seats
      WHEN oc.removed_orcid IS NOT NULL AND ar.orcid = oc.removed_orcid THEN NULL          -- remove: detach wrong ORCID, this author only
      WHEN m.works_count > 0 THEN ar.orcid                                                 -- no trusted seat evidence either way: keep the mint-time value
      ELSE NULL                                                                            -- zero works: nothing supports the legacy value, drop it
    END AS orcid,
    op.seat_orcids_kept,
    ar.full_name,
    named_struct(
      '2yr_mean_citedness', COALESCE(two_year_mean, 0),
      'h_index',
        COALESCE(
          CAST(
            ARRAY_MAX(
              ZIP_WITH(
                sorted_citations,
                SEQUENCE(1, SIZE(sorted_citations)),
                (citation, rank) -> IF(citation >= rank, rank, 0)
              )
            ) AS INT
          ), 0),
      'i10_index', COALESCE(i10_index,0)
    ) AS summary_stats
  FROM main_agg_pre m
  LEFT JOIN openalex.authors.authors ar ON m.id = ar.id
  LEFT JOIN openalex.authors.author_names_curations c ON m.id = c.author_id
  LEFT JOIN openalex.authors.author_orcid_curations oc ON m.id = oc.author_id
  LEFT JOIN observed_pre op ON m.id = op.id
  LEFT JOIN served_name sn ON m.id = sn.id
  LEFT JOIN marker_name mn ON m.id = mn.id AND mn.nm = COALESCE(sn.served_name, ar.display_name)  -- oxjob #1990
),

-- oxjob #1849: an ORCID that is another profile's primary never sits in this profile's observed_orcids.
-- The ORCID filters match observed_orcids (#1340), so one stray seat put a whole profile on someone else's
-- ORCID: Wiley's deposit of W2885272010 printed the Rennes Xavier Le Goff's ORCID (0000-0002-5297-2421,
-- primary of A5033578678) on the ICSM chemist's seat, and authorships.author.orcid:0000-0002-5297-2421 returned
-- his 600 works (ZD 26475); 21 'Peter Z. Takacs' seats printing the ELTE Peter Takacs's ORCID did the same for
-- A5101880358 (ZD 26937). Follows Jason 2026-09-19 (#1267: an ORCID is never on two profiles). A profile keeps
-- its own primary even when another profile shares it (that is #1381's merge class); a non-primary ORCID that
-- is no profile's primary stays.
primary_orcids AS (
  SELECT DISTINCT orcid FROM enriched_agg WHERE orcid IS NOT NULL AND orcid <> ''
),
observed_final AS (
  SELECT e.id,
         TRANSFORM(ARRAY_SORT(COLLECT_LIST(STRUCT(e.pos AS pos, e.orcid AS orcid))), x -> x.orcid) AS kept_orcids
  FROM (
    SELECT m.id, m.orcid AS own_orcid, s.pos, s.x.orcid AS orcid
    FROM enriched_agg m
    LATERAL VIEW POSEXPLODE(m.seat_orcids_kept) s AS pos, x
    WHERE SIZE(m.seat_orcids_kept) > 0
  ) e
  LEFT JOIN primary_orcids po ON po.orcid = e.orcid AND e.orcid <> COALESCE(e.own_orcid, '')
  WHERE po.orcid IS NULL
  GROUP BY e.id
),

-- oxjob #1107: names hidden from a profile on request (a former name: users-api curation authors /
-- display_name_alternatives / remove, synced by SyncAuthorHiddenNames). A name is hidden when its key (letter/digit
-- tokens, lowercased, sorted) is a hidden key. Only served fields change: raw_author_names (the API's
-- display_name_alternatives and raw_author_names) drops the name, and a full_name that is a hidden name serves the
-- display_name (block_key follows the served full_name). authors_for_matching (below) still matches on every attested
-- name. Revert: delete these two CTEs and read enriched_agg in the final SELECT again.
hidden_keys AS (
  SELECT author_id, COLLECT_SET(name_key) AS name_keys
  FROM openalex.authors.author_hidden_names
  GROUP BY author_id
),
served_agg AS (
  SELECT m.* EXCEPT (raw_author_names, full_name),
    CASE WHEN hk.author_id IS NULL THEN m.raw_author_names
         ELSE FILTER(m.raw_author_names, n -> NOT ARRAY_CONTAINS(hk.name_keys, ARRAY_JOIN(ARRAY_SORT(FILTER(SPLIT(LOWER(REGEXP_REPLACE(n, '[^\\p{L}\\p{N}]+', ' ')), ' '), t -> t <> '')), ' ')))
    END AS raw_author_names,
    CASE WHEN hk.author_id IS NOT NULL AND ARRAY_CONTAINS(hk.name_keys, ARRAY_JOIN(ARRAY_SORT(FILTER(SPLIT(LOWER(REGEXP_REPLACE(m.full_name, '[^\\p{L}\\p{N}]+', ' ')), ' '), t -> t <> '')), ' '))
         THEN m.display_name
         ELSE m.full_name
    END AS full_name
  FROM enriched_agg m
  LEFT JOIN hidden_keys hk ON m.id = hk.author_id
)

-- final select
SELECT
  m.id,
  -- oxjob #2106: the name text normalised (dotless i + accent, decomposed accents, invisible characters)
  -- oxjob #2346: first, the name the profile's bylines show (catalogue string or leading title cleaned; never a curated name)
  COALESCE(bnf.clean_name, ntf.clean_name, m.display_name) AS display_name,
  m.raw_author_names,
  m.orcid,
  -- every trusted ORCID on the profile's seats, primary first (#1267), minus other profiles' primaries (#1849); [] when none
  ARRAY_COMPACT(ARRAY_DISTINCT(CONCAT(ARRAY(m.orcid), COALESCE(obf.kept_orcids, ARRAY())))) AS observed_orcids,
  m.full_name,
  -- block_key derived from full_name via author_names
  CASE
    WHEN pn_bk.parsed_name.last IS NULL THEN NULL
    WHEN pn_bk.parsed_name.first IS NULL OR pn_bk.parsed_name.first = '' THEN pn_bk.parsed_name.last
    ELSE CONCAT(SUBSTRING(pn_bk.parsed_name.first, 1, 1), ' ', pn_bk.parsed_name.last)
  END AS block_key,
  m.works_count,
  m.cited_by_count,
  m.summary_stats,
  named_struct(
    'openalex', m.author_id_str,
    'orcid', m.orcid,
    'observed_orcids', ARRAY_COMPACT(ARRAY_DISTINCT(CONCAT(ARRAY(m.orcid), COALESCE(obf.kept_orcids, ARRAY()))))
  ) AS ids,
  COALESCE(ap.affiliations, ARRAY()) AS affiliations,
  la.last_known_institutions,
  COALESCE(at.topics, ARRAY()) AS topics,
  COALESCE(at.topic_share, ARRAY()) AS topic_share,
  COALESCE(cp.x_concepts, ARRAY()) AS x_concepts,
  COALESCE(s.sources, ARRAY()) AS sources,
  cby.counts_by_year,
  m.works_api_url,
  COALESCE(aw.awards_count, 0) AS awards_count,
  CONCAT('https://api.openalex.org/awards?filter=investigators.author.id:',
         REGEXP_EXTRACT(m.author_id_str, '/(A[0-9]+)$', 1)) AS awards_api_url,
  m.updated_date,
  m.created_date
FROM served_agg m
LEFT JOIN openalex.authors.author_topics at ON m.author_id_str = at.author_id
LEFT JOIN concepts_packed cp ON m.author_id_str = cp.author_id
LEFT JOIN affiliations_packed ap ON m.author_id_str = ap.author_id
LEFT JOIN sources_agg s ON m.author_id_str = s.author_id
LEFT JOIN counts_agg cby ON m.author_id_str = cby.author_id
LEFT JOIN last_inst_agg la ON m.author_id_str = la.author_id
LEFT JOIN awards_agg aw ON m.author_id_str = aw.author_id
LEFT JOIN openalex.authors.author_names pn_bk ON TRIM(m.full_name) = pn_bk.raw_author_name
-- oxjob #1849: observed ORCIDs minus other profiles' primaries
LEFT JOIN observed_final obf ON obf.id = m.id
-- oxjob #2106: ~40K names, built in the cell above
LEFT JOIN openalex.authors.author_name_text_fixes ntf ON ntf.raw_name = m.display_name
-- oxjob #2346: the byline cleaner's name, built in its cell above; a profile with a curated name keeps it as curated
LEFT JOIN (SELECT DISTINCT author_id FROM openalex.authors.author_names_curations
           WHERE NULLIF(curated_display_name, '') IS NOT NULL) cur ON cur.author_id = m.id
-- #2346 Part 2: profiles whose catalogue rename would newly flag a non-owner seat in the nightly rematch keep their name
LEFT JOIN openalex.authors.author_name_clean_holds nch ON nch.author_id = m.id
LEFT JOIN openalex.authors.author_byline_name_fixes bnf
  ON bnf.raw_name = COALESCE(ntf.clean_name, m.display_name) AND cur.author_id IS NULL AND nch.author_id IS NULL;

In [ ]:
%sql
-- Compare hashes and set updated_date only when content changed
WITH new_hashes AS (
  SELECT id,
    xxhash64(CONCAT_WS('|',
      CAST(id AS STRING),
      COALESCE(display_name, ''),
      COALESCE(full_name, ''),
      COALESCE(TO_JSON(raw_author_names), '[]'),
      COALESCE(orcid, ''),
      COALESCE(TO_JSON(observed_orcids), '[]'),
      COALESCE(TO_JSON(ids), '{}'),
      COALESCE(CAST(works_count AS STRING), '0'),
      COALESCE(CAST(cited_by_count AS STRING), '0'),
      COALESCE(TO_JSON(summary_stats), '{}'),
      COALESCE(TO_JSON(affiliations), '[]'),
      COALESCE(TO_JSON(last_known_institutions), '[]'),
      COALESCE(TO_JSON(sources), '[]'),
      COALESCE(TO_JSON(counts_by_year), '[]'),
      COALESCE(TO_JSON(topics), '[]'),
      -- topic_share excluded (non-deterministic: depends on global corpus)
      COALESCE(TO_JSON(x_concepts), '[]'),
      -- grants (oxjob #1847): NULL for a profile with none, and CONCAT_WS skips NULL, so those profiles keep their hash
      CASE WHEN COALESCE(awards_count, 0) > 0 THEN CAST(awards_count AS STRING) END
    )) AS content_hash
  FROM openalex.authors.openalex_authors
)
MERGE INTO openalex.authors.openalex_authors AS target
USING (
  SELECT n.id,
    CASE
      WHEN p.id IS NULL THEN DATE_TRUNC('SECOND', CURRENT_TIMESTAMP())
      WHEN n.content_hash <> p.content_hash THEN DATE_TRUNC('SECOND', CURRENT_TIMESTAMP())
      ELSE p.updated_date
    END AS new_updated_date
  FROM new_hashes n
  LEFT JOIN openalex.authors.openalex_authors_hash p ON n.id = p.id
) AS source
ON target.id = source.id
WHEN MATCHED THEN UPDATE SET target.updated_date = source.new_updated_date;

In [ ]:
-- Compact files after MERGE so liquid clustering on (id, updated_date)
-- organizes files by updated_date. This enables data skipping for
-- downstream jobs that filter on updated_date (snapshot export, ES sync).
OPTIMIZE openalex.authors.openalex_authors;

### Create mapping table for AND

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.authors_for_matching
CLUSTER BY (block_key)
AS
SELECT
  -- 1. Blocking Key: the profile-level canonical parse (oxjob #1360) when the profile has one, else the full_name parse
  -- oxjob #1107: a profile with a hidden name matches on its stored full_name (openalex_authors serves the display name)
  COALESCE(cn.block_key, CASE WHEN hid.id IS NULL THEN oa.block_key
                         WHEN pn.parsed_name.last IS NULL THEN NULL
                         WHEN pn.parsed_name.first IS NULL OR pn.parsed_name.first = '' THEN pn.parsed_name.last
                         ELSE CONCAT(SUBSTRING(pn.parsed_name.first, 1, 1), ' ', pn.parsed_name.last) END) AS block_key,

  -- 2. Candidate Identity
  oa.id AS author_id,
  oa.display_name,
  COALESCE(hid.full_name, oa.full_name) AS full_name,
  oa.ids.orcid as orcid,
  oa.observed_orcids,  -- every trusted ORCID on the profile; MatchAuthors' ORCID tier explodes this (#1267)

  -- 3. Parsed name columns for matching in MatchAuthors: canonical (voted over every attested string, oxjob #1360) else full_name's
  COALESCE(cn.first_norm, pn.parsed_name.first) AS first,
  SUBSTRING(COALESCE(cn.first_norm, pn.parsed_name.first), 1, 1) AS first_initial,
  COALESCE(cn.middle_norm, pn.parsed_name.middle) AS middle,
  COALESCE(cn.last_norm, pn.parsed_name.last) AS last,
  -- alternate readings (oxjob #1341 generator + the canonical family_tokens, oxjob #1360)
  COALESCE(cn.alt_parses, pn.alt_parses) AS alt_parses,
  COALESCE(TRANSFORM(cn.alt_parses, x -> x.key), pn.alt_block_keys) AS alt_block_keys,
  cn.confidence AS parse_confidence,
  cn.order_source AS parse_order_source,
  cn.parse_version AS parse_version,

  ARRAY_DISTINCT(
     TRANSFORM(
       FILTER(oa.affiliations, x -> x.institution.id IS NOT NULL),
       x -> x.institution.id
     )
  ) AS institution_ids,
  
  ARRAY_DISTINCT(
     TRANSFORM(oa.topics, x -> x.id)
  ) AS topic_ids,

  array_distinct(
    TRANSFORM(oa.topics, x -> x.subfield.id)
  ) AS subfield_ids,

  TRANSFORM(oa.sources, x -> x.id) AS source_ids,

  ARRAY_DISTINCT(
    TRANSFORM(
        -- oxjob #1107: hidden names still count for matching (the profile's unfiltered names)
        ARRAY_UNION(ARRAY(oa.display_name), COALESCE(hid.all_names, oa.raw_author_names, ARRAY())),
        n -> LOWER(TRIM(n))
    )
  ) AS name_variants,

  oa.works_count,
  oa.cited_by_count,
  COALESCE(oa.summary_stats.h_index, 0) AS h_index,
  
  -- first/last publication year of the profile's works. Read from the works, not counts_by_year: since oxjob #1878
  -- counts_by_year also lists years with citations but no papers and stops at the current year. Same values as before.
  ay.first_active_year,
  ay.last_active_year

FROM openalex.authors.openalex_authors oa
-- oxjob #1107: profiles with a hidden name (a few rows): their stored full_name and every attested name, built as
-- openalex_authors' raw_author_names is before the hide (16 Nov: one curation hides every name with the same key, so
-- the curation texts alone missed same-key variants).
LEFT JOIN (
  SELECT h.author_id AS id, ar.full_name,
         SLICE(ARRAY_SORT(ARRAY_DISTINCT(ARRAY_COMPACT(
           CONCAT(COLLECT_LIST(b.author_display_name), COLLECT_LIST(b.raw_author_name))))), 1, 100) AS all_names
  FROM (SELECT DISTINCT author_id FROM openalex.authors.author_hidden_names) h
  LEFT JOIN openalex.authors.authors ar ON ar.id = h.author_id
  JOIN openalex.authors.author_works_staging b ON b.author_id = CONCAT('https://openalex.org/A', CAST(h.author_id AS STRING))
  GROUP BY h.author_id, ar.full_name
) hid ON hid.id = oa.id
LEFT JOIN openalex.authors.author_names pn ON TRIM(COALESCE(hid.full_name, oa.full_name)) = pn.raw_author_name
LEFT JOIN openalex.authors.author_canonical_names cn
  ON cn.author_id = oa.id AND cn.kind = 'person' AND cn.block_key IS NOT NULL AND cn.block_key <> ''
LEFT JOIN (
  SELECT CAST(REPLACE(author_id, 'https://openalex.org/A', '') AS BIGINT) AS author_id,
         CAST(MIN(pub_year) AS INT) AS first_active_year,
         CAST(MAX(pub_year) AS INT) AS last_active_year
  FROM openalex.authors.author_works_staging
  WHERE pub_year IS NOT NULL
  GROUP BY author_id
) ay ON ay.author_id = oa.id
WHERE COALESCE(cn.block_key, CASE WHEN hid.id IS NULL THEN oa.block_key
                         WHEN pn.parsed_name.last IS NULL THEN NULL
                         WHEN pn.parsed_name.first IS NULL OR pn.parsed_name.first = '' THEN pn.parsed_name.last
                         ELSE CONCAT(SUBSTRING(pn.parsed_name.first, 1, 1), ' ', pn.parsed_name.last) END) IS NOT NULL
  -- Exclude zero-works profiles from the match candidate pool. Drains the merged-loser tail:
  -- profiles whose work_authors rows have all been rewritten to a winner_author_id by any
  -- author_merge_log_* MERGE step show works_count = 0 here on the next CreateAuthors
  -- refresh, so this filter prevents MatchAuthors from re-attaching new daily-ingest works
  -- to drained losers (race-tail problem observed at 24h Phase 4 drain).
  -- Also drops ~3M legacy zero-works profiles (junk metadata, non-person entities,
  -- abandoned MAG/Crossref records) that have no business being match candidates.
  -- Match-candidate pool: 117.7M -> ~116.4M (-3.2%, all zero-signal profiles).
  AND oa.works_count > 0;

OPTIMIZE openalex.authors.authors_for_matching;

-- oxjob #1341: one row per (profile, key) over the primary block key plus every alternate key.
-- MatchAuthors joins a seat's keys (primary + alternates) against this table to reach profiles that
-- sit in another primary block (compound surnames, reversed order, transliteration variants).
-- Alternate keys never extend into mega blocks: an alternate entry is kept only when fewer than
-- 10K profiles carry that key; primary entries are always kept. Served names never read this.
CREATE OR REPLACE TABLE openalex.authors.authors_for_matching_keys
CLUSTER BY (key)
AS
WITH all_keys AS (
  SELECT author_id, block_key AS key, TRUE AS is_primary, first, last, first AS p_first, last AS p_last
  FROM openalex.authors.authors_for_matching
  WHERE block_key IS NOT NULL AND block_key != ''
  UNION ALL
  -- p_first/p_last = the profile's primary parse, so MatchAuthors can classify the reading (oxjob #1341)
  SELECT author_id, p.key, FALSE AS is_primary, p.first, p.last, first AS p_first, last AS p_last
  FROM openalex.authors.authors_for_matching
  LATERAL VIEW explode(COALESCE(alt_parses, ARRAY())) AS p
),
dedup AS (
  -- one row per (profile, key); a primary entry wins over an alternate with the same key
  SELECT author_id, key, MAX(is_primary) AS is_primary,
         MAX_BY(first, is_primary) AS first, MAX_BY(last, is_primary) AS last, MAX(p_first) AS p_first, MAX(p_last) AS p_last
  FROM all_keys GROUP BY author_id, key
),
sizes AS (
  SELECT key, COUNT(*) AS block_size FROM dedup GROUP BY key
)
SELECT d.author_id, d.key, d.is_primary, d.first, d.last, d.p_first, d.p_last, CAST(s.block_size AS INT) AS block_size
FROM dedup d
JOIN sizes s ON s.key = d.key
WHERE d.is_primary OR s.block_size <= 10000;

OPTIMIZE openalex.authors.authors_for_matching_keys;